<a href="https://colab.research.google.com/github/dankim0328/revisit-analysis/blob/main/recsys2019_SSM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd

# 드라이브 마운트 후 경로 재설정
path = '/content/drive/MyDrive/Recsys2019/'

# 상위 5행만 읽어오기
train_head = pd.read_csv(path + 'train.csv', nrows=5)
item_metadata_head = pd.read_csv(path + 'item_metadata.csv', nrows=5)

print("--- Train Data Preview ---")
display(train_head)

print("\n--- Item Metadata Preview ---")
display(item_metadata_head)

--- Train Data Preview ---


,user_id,session_id,timestamp,step,action_type,reference,platform,city,device,current_filters,impressions,prices
0,00RL8Z82B2Z1,aff3928535f48,1541037460,1,search for poi,Newtown,AU,"Sydney, Australia",mobile,NaN,NaN,NaN
1,00RL8Z82B2Z1,aff3928535f48,1541037522,2,interaction item image,666856,AU,"Sydney, Australia",mobile,NaN,NaN,NaN
2,00RL8Z82B2Z1,aff3928535f48,1541037522,3,interaction item image,666856,AU,"Sydney, Australia",mobile,NaN,NaN,NaN
3,00RL8Z82B2Z1,aff3928535f48,1541037532,4,interaction item image,666856,AU,"Sydney, Australia",mobile,NaN,NaN,NaN
4,00RL8Z82B2Z1,aff3928535f48,1541037532,5,interaction item image,109038,AU,"Sydney, Australia",mobile,NaN,NaN,NaN



--- Item Metadata Preview ---


,item_id,properties
0,5101,Satellite TV|Golf Course|Airport Shuttle|Cosme...
1,5416,Satellite TV|Cosmetic Mirror|Safe (Hotel)|Tele...
2,5834,Satellite TV|Cosmetic Mirror|Safe (Hotel)|Tele...
3,5910,Satellite TV|Sailing|Cosmetic Mirror|Telephone...
4,6066,Satellite TV|Sailing|Diving|Cosmetic Mirror|Sa...


In [6]:
# 1. 아이템 관련 액션만 필터링
item_actions = train[train['reference'].str.isnumeric() == True].copy()
item_actions['reference'] = item_actions['reference'].astype(str)

# 2. 유저별로 그룹화하여 세션 간 이동 확인
def analyze_user_revisits(group):
    # 시간순 정렬
    group = group.sort_values('timestamp')

    seen_items = set()
    cross_session_revisit = 0
    sessions = group['session_id'].unique()

    # 첫 번째 세션에서 본 아이템들 저장
    first_session_items = set(group[group['session_id'] == sessions[0]]['reference'])

    # 두 번째 세션 이후부터, 이전 세션들에서 봤던 아이템을 또 보는지 확인
    total_revisits = 0
    if len(sessions) > 1:
        for i in range(1, len(sessions)):
            current_session_items = set(group[group['session_id'] == sessions[i]]['reference'])
            previous_items = set(group[group['session_id'].isin(sessions[:i])]['reference'])

            # 교집합 확인 (이전 세션에서 봤던 걸 현재 세션에서 또 봤는가?)
            revisited_in_this_session = current_session_items.intersection(previous_items)
            total_revisits += len(revisited_in_this_session)

    return pd.Series({
        'total_sessions': len(sessions),
        'cross_session_revisit_count': total_revisits,
        'is_multi_session_user': 1 if len(sessions) > 1 else 0
    })

user_stats = item_actions.groupby('user_id').apply(analyze_user_revisits)

print("--- 유저 단위 세션 간 재방문 분석 ---")
multi_session_users = user_stats[user_stats['is_multi_session_user'] == 1]
print(f"2개 이상의 세션을 가진 유저 수: {len(multi_session_users)}")
print(f"재접속 후 이전 세션 아이템을 다시 본 유저 비율: {(multi_session_users['cross_session_revisit_count'] > 0).mean() * 100:.2f}%")

--- 유저 단위 세션 간 재방문 분석 ---
2개 이상의 세션을 가진 유저 수: 892
재접속 후 이전 세션 아이템을 다시 본 유저 비율: 35.43%


/tmp/ipykernel_680/2477513329.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_stats = item_actions.groupby('user_id').apply(analyze_user_revisits)


In [11]:
import pandas as pd

path = '/content/drive/MyDrive/Recsys2019/'

# 1. 메모리 절약을 위해 데이터 타입을 지정하고 필요한 컬럼만 로드
# nrows 옵션이 있다면 반드시 제거해야 전체 데이터를 읽습니다.
t_dtypes = {
    'user_id': 'str',
    'session_id': 'str',
    'action_type': 'str',
    'reference': 'str',
    'timestamp': 'int64'
}

print("데이터 로딩 시작 (전체 행을 처리합니다)...")

# chunksize를 사용하여 메모리 초과 방지
chunks = pd.read_csv(path + 'train.csv',
                     usecols=['user_id', 'session_id', 'action_type', 'reference', 'timestamp'],
                     dtype=t_dtypes,
                     chunksize=500000) # 50만 행씩 끊어서 읽기

full_item_actions = []

for i, chunk in enumerate(chunks):
    # 아이템 관련 액션이고 reference가 숫자인 것만 남기고 즉시 필터링
    temp = chunk.dropna(subset=['reference'])
    temp = temp[temp['reference'].str.isnumeric()]
    full_item_actions.append(temp)
    if i % 5 == 0:
        print(f"{i*500000:,} 행 처리 중...")

# 하나로 합치기
item_actions = pd.concat(full_item_actions)
print(f"필터링 완료! 총 아이템 액션 수: {len(item_actions):,}")

# 2. 분석 함수 (경고 메시지 해결을 위해 include_groups=False 대응)
def analyze_user_revisits(group):
    # group은 이제 user_id 컬럼을 포함하지 않는 DataFrame이 됩니다 (pandas 최신버전 대응)
    group = group.sort_values('timestamp')
    sessions = group['session_id'].unique()

    total_revisits = 0
    if len(sessions) > 1:
        session_item_sets = [set(group[group['session_id'] == s]['reference']) for s in sessions]
        accumulated_previous_items = set()

        for i in range(1, len(sessions)):
            accumulated_previous_items.update(session_item_sets[i-1])
            current_session_items = session_item_sets[i]
            revisited_in_this_session = current_session_items.intersection(accumulated_previous_items)
            total_revisits += len(revisited_in_this_session)

    return pd.Series({
        'total_sessions': len(sessions),
        'cross_session_revisit_count': total_revisits,
        'is_multi_session_user': 1 if len(sessions) > 1 else 0
    })

# 3. 그룹화 실행 (경고 해결을 위해 include_groups=False 추가)
print("유저별 재방문 분석 중...")
user_stats = item_actions.groupby('user_id').apply(analyze_user_revisits, include_groups=False)

# 4. 결과 출력
multi_session_users = user_stats[user_stats['is_multi_session_user'] == 1]
print("\n--- 전체 데이터 기반 유저 단위 재방문 분석 ---")
print(f"전체 유저 수: {len(user_stats):,}")
print(f"2개 이상의 세션을 가진 유저 수: {len(multi_session_users):,}")
if len(multi_session_users) > 0:
    print(f"재접속 후 이전 세션 아이템을 다시 본 유저 비율: {(multi_session_users['cross_session_revisit_count'] > 0).mean() * 100:.2f}%")

데이터 로딩 시작 (전체 행을 처리합니다)...
0 행 처리 중...
2,500,000 행 처리 중...
5,000,000 행 처리 중...
7,500,000 행 처리 중...
10,000,000 행 처리 중...
12,500,000 행 처리 중...
15,000,000 행 처리 중...
필터링 완료! 총 아이템 액션 수: 14,295,333
유저별 재방문 분석 중...

--- 전체 데이터 기반 유저 단위 재방문 분석 ---
전체 유저 수: 725,968
2개 이상의 세션을 가진 유저 수: 108,822
재접속 후 이전 세션 아이템을 다시 본 유저 비율: 46.06%


In [12]:
# 구글 드라이브의 본인 폴더에 저장
item_actions.to_csv(path + 'train_filtered_revisitation.csv', index=False)

print("저장 완료! 이제 다음 세션부터는 이 파일만 불러오면 됩니다.")

저장 완료! 이제 다음 세션부터는 이 파일만 불러오면 됩니다.


In [14]:
# 1. 변수명 통일 (기존 item_actions를 train_ready로 복사)
train_ready = item_actions.copy()

# 2. 정렬 (유저별 -> 아이템별 -> 시간순)
# Delta t (재방문 간격)를 계산하기 위해 필수적인 단계입니다.
train_ready = train_ready.sort_values(['user_id', 'reference', 'timestamp'])

# 3. Delta t 계산 (단위: 시간)
# 동일 유저가 동일 아이템(reference)을 이전에 언제 봤는지 계산합니다.
train_ready['prev_timestamp'] = train_ready.groupby(['user_id', 'reference'])['timestamp'].shift(1)
train_ready['delta_t'] = (train_ready['timestamp'] - train_ready['prev_timestamp']) / 3600

# 4. 확인
print(train_ready[train_ready['delta_t'].notnull()][['user_id', 'reference', 'delta_t']].head())

              user_id reference   delta_t
8321583  0001VQMGUI65   3133074  0.005556
8321584  0001VQMGUI65   3133074  0.005278
8321585  0001VQMGUI65   3133074  0.001667
8321586  0001VQMGUI65   3133074  0.002778
8321587  0001VQMGUI65   3133074  0.000000


In [15]:
from scipy.stats import norm

# k 값의 범위에 따른 m(k) 테이블 생성
k_grid = np.linspace(-5, 5, 1000)
m_table = norm.pdf(k_grid) - k_grid * (1 - norm.cdf(k_grid))

def get_m(k):
    # k_grid에서 가장 가까운 인덱스를 찾아 m(k) 반환
    idx = np.searchsorted(k_grid, k)
    return m_table[np.clip(idx, 0, len(m_table)-1)]

In [16]:
def simulate_likelihood(params, user_data, D=50):
    alpha, beta, rho, c, sigma_eta = params
    total_log_likelihood = 0

    for d in range(D):
        # 유저별 초기 epsilon_d 생성
        epsilon_latent = {}
        current_user_lik = 1.0

        for row in user_data.itertuples():
            item = row.reference
            dt = row.delta_t

            # 1. Expected Utility 계산
            if item in epsilon_latent:
                # 재방문인 경우 AR(1) 적용
                expected_epsilon = (rho ** dt) * epsilon_latent[item]
                sigma_t = np.sqrt(sigma_eta**2 * (1 - rho**(2*dt)) / (1 - rho**2))
            else:
                # 첫 검색
                expected_epsilon = 0
                sigma_t = sigma_eta

            # 2. Reservation Utility (z_ijt) 계산
            # z = expected_utility + option_value
            z = (row.delta_ij + expected_epsilon) + sigma_t * get_m(c / sigma_t)

            # 3. Kernel-smoothed Likelihood (Multivariate Logistic)
            # Action에 따른 확률 계산 (예: 클릭 확률)
            prob = 1 / (1 + np.exp(-(z - threshold))) # 단순화된 예시
            current_user_lik *= prob

            # 4. State Update: 실제 시뮬레이션된 새로운 epsilon 저장
            new_eta = np.random.normal(0, sigma_eta)
            epsilon_latent[item] = expected_epsilon + new_eta

        total_log_likelihood += current_user_lik

    return np.log(total_log_likelihood / D)

In [17]:
def extract_clicked_price(row):
    try:
        if pd.isna(row['impressions']) or pd.isna(row['prices']):
            return np.nan
        imps = row['impressions'].split('|')
        pris = row['prices'].split('|')
        if row['reference'] in imps:
            idx = imps.index(row['reference'])
            return float(pris[idx])
    except:
        return np.nan
    return np.nan

# 전체 데이터에 적용 (시간이 다소 소요될 수 있습니다)
print("가격 정보 추출 중...")
# 앞서 만든 train_ready에 가격 정보를 추가합니다.
# 원본 train 데이터에서 impressions와 prices 컬럼을 다시 가져와야 할 수 있습니다.
# train_ready = train_ready.merge(train[['user_id', 'session_id', 'timestamp', 'step', 'impressions', 'prices']], on=...)

train_ready['price'] = train_ready.apply(extract_clicked_price, axis=1)

가격 정보 추출 중...


In [18]:
from collections import Counter

# 1. 가장 빈번한 속성 20개 추출
all_props = []
item_metadata['properties'].dropna().apply(lambda x: all_props.extend(x.split('|')))
top_20_props = [p for p, count in Counter(all_props).most_common(20)]

# 2. Multi-hot Encoding
for prop in top_20_props:
    item_metadata[f'prop_{prop}'] = item_metadata['properties'].apply(
        lambda x: 1 if isinstance(x, str) and prop in x else 0
    )

# 3. 필요한 컬럼만 추출
item_features = item_metadata[['item_id'] + [f'prop_{p}' for p in top_20_props]]
item_features['item_id'] = item_features['item_id'].astype(str)

# 4. train_ready와 결합
train_ready = train_ready.merge(item_features, left_on='reference', right_on='item_id', how='left')

NameError: name 'item_metadata' is not defined

In [19]:
def objective_function(params):
    # params: [alpha(가격), beta1...beta20(속성), rho(지속성), c(검색비용), sigma_eta(충격)]
    alpha = params[0]
    betas = params[1:21]
    rho = params[21]
    c = params[22]
    sigma_eta = params[23]

    # delta_ij 계산 (Intrinsic Utility)
    # delta_ij = alpha * price + sum(beta_k * prop_k)

    # 10만 명을 다 돌리면 너무 오래 걸리니,
    # Pilot Study를 위해 유저 1,000명만 샘플링해서 돌리는 것을 추천합니다.
    sample_users = train_ready['user_id'].unique()[:1000]
    sample_data = train_ready[train_ready['user_id'].isin(sample_users)]

    # log_likelihood = simulate_likelihood(...) 계산
    # return -log_likelihood (Scipy minimize는 최소화를 수행하므로)
    pass

In [ ]:
import pandas as pd
from collections import Counter

# 1. 파일 다시 로드 (경로는 본인의 설정에 맞춰주세요)
path = '/content/drive/MyDrive/Recsys2019/'
item_metadata = pd.read_csv(path + 'item_metadata.csv')

# 2. 가장 빈번한 속성 20개 추출
all_props = []
item_metadata['properties'].dropna().apply(lambda x: all_props.extend(x.split('|')))
top_20_props = [p for p, count in Counter(all_props).most_common(20)]

# 3. Multi-hot Encoding
for prop in top_20_props:
    # 컬럼명에 공백이 있을 수 있으니 안전하게 처리
    col_name = f"prop_{prop.replace(' ', '_')}"
    item_metadata[col_name] = item_metadata['properties'].apply(
        lambda x: 1 if isinstance(x, str) and prop in x else 0
    )

# 4. 필요한 컬럼만 추출 (item_id와 생성한 속성들)
prop_cols = [f"prop_{p.replace(' ', '_')}" for p in top_20_props]
item_features = item_metadata[['item_id'] + prop_cols].copy()
item_features['item_id'] = item_features['item_id'].astype(str)

# 5. 기존 train_ready와 결합
# (train_ready에 이미 item_id 컬럼이 있다면 중복 방지를 위해 drop 후 결합)
if 'item_id' in train_ready.columns:
    train_ready = train_ready.drop(columns=['item_id'])

train_ready = train_ready.merge(item_features, left_on='reference', right_on='item_id', how='left')

print("아이템 속성 결합 완료!")
print(f"결합 후 컬럼 수: {len(train_ready.columns)}")

In [ ]:
# 재방문(delta_t가 있는) 경험이 있는 유저들 중 1,000명 샘플링
revisit_users = train_ready[train_ready['delta_t'].notnull()]['user_id'].unique()
sample_user_ids = revisit_users[:1000]

sample_df = train_ready[train_ready['user_id'].isin(sample_user_ids)].copy()

print(f"샘플링된 유저 수: {len(sample_user_ids)}")
print(f"샘플링된 행 수: {len(sample_df)}")